# S1-M2-01 — Sensibilidad del etiquetador al ruido

**Alejandro Zamora · M2 · rama `feat/m2-ruido`**

La pregunta: el etiquetador encuentra los giros perfectamente en una serie limpia.
Con ruido, ¿cuánto tolera antes de empezar a inventar giros que no existen?

Se puede responder porque en `serie_zigzag` **los giros los ponemos nosotros**: la verdad
de referencia no sale de `etiquetar()`, sale de los vértices con los que se construyó la serie.

> **Esta serie no es Litecoin.** Es lineal a tramos con ruido gaussiano: no tiene
> heterocedasticidad, ni colas pesadas, ni saltos. Los números de acá caracterizan al
> **etiquetador**, no al mercado.

La lógica vive en `src/sintetico/sensibilidad.py` con sus pruebas en `tests/test_sensibilidad.py`.
Este notebook solo explora; la evidencia del informe se regenera con:

```bash
uv run python -m src.sintetico.sensibilidad
```


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.sintetico.sensibilidad import (
    NIVELES_FINOS,
    figura_curvas,
    figura_series_ejemplo,
    medir_sensibilidad,
    punto_de_quiebre,
    resumir,
)
from src.visual import estilo

estilo.aplicar()
pd.set_option('display.width', 200)


## Paso 1 — El caso limpio

Antes de medir degradación hay que verificar que sin ruido la detección es perfecta.
Si no lo fuera, el punto de quiebre que midamos no sería del ruido: sería del etiquetador.


In [ ]:
limpio = medir_sensibilidad(niveles_ruido=[0.0], semillas=range(10))
print('recall exacto:', limpio['recall_exacto'].unique())
print('falsos positivos:', limpio['falsos_positivos'].sum())
print('giros verdaderos totales:', limpio['verdaderos'].sum())


## Paso 2 — Barrido principal

Siete niveles de ruido, diez semillas cada uno. Se promedia sobre semillas porque una sola
serie no distingue el efecto del ruido del azar de esa serie concreta.

El eje que importa es **`ruido_relativo` = sigma / cambio típico por vela**: sigma en unidades
absolutas no significa nada por sí solo.


In [ ]:
crudo = medir_sensibilidad()
resumen = resumir(crudo)
resumen


## Paso 3 — El punto de quiebre

El criterio está fijado en código (`punto_de_quiebre`), no elegido mirando la tabla.


In [ ]:
punto_de_quiebre(resumen)


### Barrido fino

El salto entre sigma 0,5 y 1,0 es grande. Un segundo barrido más denso localiza mejor
dónde aparece el primer giro falso.


In [ ]:
resumen_fino = resumir(medir_sensibilidad(niveles_ruido=NIVELES_FINOS))
resumen_fino


In [ ]:
punto_de_quiebre(resumen_fino)


## Paso 4 — Figuras

No se guardan desde acá: `generar_evidencia()` las deja en `docs/evidencias/` con su
`.generado.txt` al lado.


In [ ]:
figura_curvas(resumen, resumen_fino);


In [ ]:
figura_series_ejemplo();


## Conclusión

Completar con los números que salgan de la corrida de arriba. Lo que hay que separar:

1. **Hasta qué ruido relativo la detección es exacta** (ni un giro perdido, ni uno inventado).
2. **Dónde aparece el primer giro falso** — es el umbral que importa para las etiquetas.
3. **La brecha entre detección exacta y con tolerancia de una vela**: el etiquetador
   pierde la vela exacta mucho antes de perder el giro. Son dos fallos distintos.

Y decirlo bien: esto caracteriza al etiquetador sobre una serie que construimos nosotros.
No es una medición sobre LTC.
